# Оценка качества генерации текста LLM с помощью метрик

## Трек A: Question Answering (QA)

**Обоснование выбора трека.** QA выбран как задача с чёткими эталонными ответами, что позволяет объективно сравнить модели через количественные метрики (F1, EM). Датасет SberQuAD на русском языке даёт возможность оценить кросс-лингвальные способности моделей в практически значимом сценарии.

| Модель | Тип | Описание |
|--------|-----|----------|
| **Qwen 3.5 9B** | Локальная (llama.cpp) | Квантизация Q4_K_M, Docker, GPU |
| **GPT-4o** | Облачная (OpenAI API) | Коммерческая модель |

**Датасет:** SberQuAD ([HuggingFace](https://huggingface.co/datasets/kuznetsoffandrey/sberquad)) — extractive QA, 100 примеров из validation split, seed=42.

## 1. Импорт библиотек и настройка окружения

In [ ]:
import os
import time
import re
import string
from collections import Counter
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

from datasets import load_dataset
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sacrebleu.metrics import BLEU
from scipy import stats
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

load_dotenv()

SEED = 42
np.random.seed(SEED)

print("Библиотеки импортированы.")

## 2. Подключение к моделям

Оба клиента используют OpenAI-совместимый API. Qwen 3.5 9B обслуживается локально через llama.cpp (порт 8080), GPT-4o — через OpenAI API.

In [ ]:
local_client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
cloud_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-key-here"))

MODELS = {
    "qwen3.5-9b": {
        "client": local_client,
        "model_id": "qwen3.5-9b",
        "description": "Qwen 3.5 9B (Q4_K_M, локальный llama.cpp)"
    },
    "gpt-4o": {
        "client": cloud_client,
        "model_id": "gpt-4o",
        "description": "GPT-4o (OpenAI API)"
    }
}

for name, cfg in MODELS.items():
    print(f"  {name}: {cfg['description']}")

In [ ]:
for name, config in MODELS.items():
    try:
        response = config["client"].chat.completions.create(
            model=config["model_id"],
            messages=[{"role": "user", "content": "Ответь одним словом: столица России?"}],
            max_tokens=20, temperature=0.0
        )
        print(f"[OK] {name}: {response.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"[FAIL] {name}: {e}")

## 3. Загрузка и подготовка датасета

**SberQuAD** — русскоязычный аналог SQuAD. Формат: контекст + вопрос → ответ (подстрока из контекста). Берём 100 случайных примеров из validation split для оценки.

In [ ]:
dataset = load_dataset("kuznetsoffandrey/sberquad")

print("Размеры split'ов:")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} примеров")

sample = dataset["validation"][0]
print(f"\nПример:")
print(f"  Контекст: {sample['context'][:200]}...")
print(f"  Вопрос: {sample['question']}")
print(f"  Ответы: {sample['answers']}")

In [ ]:
val_data = dataset["validation"]
indices = np.random.RandomState(SEED).choice(len(val_data), size=100, replace=False)
sampled = val_data.select(indices)

records = []
for item in sampled:
    records.append({
        "id": item["id"],
        "title": item["title"],
        "context": item["context"],
        "question": item["question"],
        "reference_answer": item["answers"]["text"][0],
        "all_answers": item["answers"]["text"],
        "context_length": len(item["context"]),
        "question_length": len(item["question"]),
        "answer_length": len(item["answers"]["text"][0])
    })

df_dataset = pd.DataFrame(records)
print(f"Подготовлено {len(df_dataset)} примеров")
print(f"\nСтатистика длин контекстов:")
print(df_dataset['context_length'].describe().round(0))
df_dataset[["question", "reference_answer", "context_length"]].head(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_dataset['context_length'], bins=20, edgecolor='black', color='steelblue')
axes[0].set_title('Длина контекста (символы)')
axes[0].set_xlabel('Символы')
axes[0].set_ylabel('Количество')

axes[1].hist(df_dataset['question_length'], bins=20, edgecolor='black', color='coral')
axes[1].set_title('Длина вопроса (символы)')
axes[1].set_xlabel('Символы')

axes[2].hist(df_dataset['answer_length'], bins=20, edgecolor='black', color='mediumseagreen')
axes[2].set_title('Длина эталонного ответа (символы)')
axes[2].set_xlabel('Символы')

plt.suptitle('Распределения длин в выборке (100 примеров)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plots/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Реализация метрик

| Метрика | Тип | Что оценивает |
|---------|-----|---------------|
| **Exact Match** | Лексическая | Бинарное совпадение после нормализации |
| **Token F1** | Лексическая | Пересечение множеств токенов (precision/recall) |
| **BLEU** | N-граммная | Совпадение последовательностей слов |
| **Semantic Similarity** | Семантическая | Косинусная близость эмбеддингов (paraphrase-multilingual-MiniLM-L12-v2) |
| **Answer Containment** | Кастомная (бонус) | Доля токенов эталона, содержащихся в ответе модели |

Нормализация: lowercase, удаление пунктуации (включая «»—–), схлопывание пробелов. Для SberQuAD используем max-over-answers (максимум метрики по всем допустимым ответам).

In [ ]:
def normalize_answer_ru(text: str) -> str:
    """Нормализация текста: lowercase, удаление пунктуации, схлопывание пробелов."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation + '«»—–„"'))
    text = ' '.join(text.split())
    return text.strip()


def get_tokens(text: str) -> List[str]:
    """Токенизация нормализованного текста по пробелам."""
    return normalize_answer_ru(text).split()


print("Нормализация:")
print(f"  'Москва — столица России!' → '{normalize_answer_ru('Москва — столица России!')}'")
print(f"  Токены: {get_tokens('Москва — столица России!')}")

In [ ]:
def exact_match(prediction: str, reference: str) -> float:
    """Exact Match — бинарное совпадение после нормализации."""
    return float(normalize_answer_ru(prediction) == normalize_answer_ru(reference))


def token_f1_score(prediction: str, reference: str) -> float:
    """Token-level F1 Score (SQuAD-стиль) — гармоническое среднее precision и recall по токенам."""
    pred_tokens = get_tokens(prediction)
    ref_tokens = get_tokens(reference)
    
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_common = sum(common.values())
    
    if num_common == 0:
        return 0.0
    
    precision = num_common / len(pred_tokens)
    recall = num_common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_bleu(prediction: str, reference: str) -> float:
    """BLEU с effective_order=True (для коротких текстов). Возвращает 0–1."""
    bleu = BLEU(effective_order=True)
    result = bleu.corpus_score([prediction], [[reference]])
    return result.score / 100.0


def max_over_answers(metric_fn, prediction: str, all_answers: List[str]) -> float:
    """Максимум метрики по всем допустимым ответам (стандарт SQuAD)."""
    return max(metric_fn(prediction, ref) for ref in all_answers)


# Валидация
print("Проверка метрик:")
print(f"  EM('Москва', 'Москва')  = {exact_match('Москва', 'Москва')}")
print(f"  EM('Москва', 'москва')  = {exact_match('Москва', 'москва')}")
print(f"  EM('Москва', 'Питер')   = {exact_match('Москва', 'Питер')}")
print(f"  F1('столица России Москва', 'Москва') = {token_f1_score('столица России Москва', 'Москва'):.3f}")
print(f"  BLEU('Москва', 'Москва') = {compute_bleu('Москва', 'Москва'):.3f}")

In [ ]:
print("Загружаем модель для семантической близости...")
st_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Модель загружена.")


def semantic_similarity(prediction: str, reference: str) -> float:
    """Косинусная близость эмбеддингов двух текстов."""
    embeddings = st_model.encode([prediction, reference])
    cos_sim = np.dot(embeddings[0], embeddings[1]) / (
        np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])
    )
    return float(cos_sim)


print(f"  sim('Москва', 'столица России') = {semantic_similarity('Москва', 'столица России'):.3f}")
print(f"  sim('Москва', 'квантовая механика') = {semantic_similarity('Москва', 'квантовая механика'):.3f}")

In [ ]:
def answer_containment(prediction: str, reference: str) -> float:
    """Кастомная метрика (бонус): доля токенов эталона, содержащихся в ответе модели.
    Полезна для выявления verbose-ответов, где правильный ответ «утоплен» в лишнем тексте."""
    pred_tokens = get_tokens(prediction)
    ref_tokens = get_tokens(reference)
    
    if not ref_tokens:
        return 1.0
    if not pred_tokens:
        return 0.0
    
    pred_set = set(pred_tokens)
    contained = sum(1 for t in ref_tokens if t in pred_set)
    return contained / len(ref_tokens)


def strip_thinking(text: str) -> str:
    """Удаление блоков <think>...</think> из вывода Qwen 3.5 (режим thinking по умолчанию)."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


print(f"Answer Containment('Система создана в 1869 году', '1869 году') = "
      f"{answer_containment('Система создана в 1869 году', '1869 году'):.3f}")

In [ ]:
def compute_all_metrics(prediction: str, reference: str,
                        all_answers: List[str], gen_time: float) -> Dict:
    """Вычисляет все метрики для одного предсказания."""
    prediction_clean = strip_thinking(prediction)
    
    return {
        "exact_match": max_over_answers(exact_match, prediction_clean, all_answers),
        "f1_score": max_over_answers(token_f1_score, prediction_clean, all_answers),
        "bleu": max_over_answers(compute_bleu, prediction_clean, all_answers),
        "semantic_similarity": max(
            semantic_similarity(prediction_clean, ref) for ref in all_answers
        ),
        "generation_time": gen_time,
        "response_length_chars": len(prediction_clean),
        "response_length_tokens": len(prediction_clean.split()),
        "answer_containment": max_over_answers(
            answer_containment, prediction_clean, all_answers
        ),
    }

## 5. Промпт-шаблоны

Три стратегии промптинга:
- **Zero-shot** — контекст + вопрос без примеров
- **Few-shot (3 примера)** — три пары вопрос-ответ перед основным запросом (примеры не из тестовой выборки)

Два дополнительных формата для A/B-теста:
- **Format A (XML)** — структурированные теги `<context>`, `<question>`
- **Format B (инструкция)** — естественная формулировка задачи

In [ ]:
SYSTEM_PROMPT_ZS = (
    "Ты — система для ответов на вопросы. "
    "Отвечай кратко, только самой сутью ответа, без объяснений."
)

def make_zero_shot_prompt(context: str, question: str) -> List[Dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT_ZS},
        {"role": "user", "content": (
            f"Контекст:\n{context}\n\n"
            f"Вопрос: {question}\n\n"
            f"Ответ:"
        )}
    ]


# Few-shot примеры (не из тестовой выборки — без data leakage)
FEW_SHOT_EXAMPLES = [
    {
        "context": "Москва — столица России, крупнейший по численности населения город России.",
        "question": "Какой город является столицей России?",
        "answer": "Москва"
    },
    {
        "context": "Байкал — озеро тектонического происхождения в южной части Восточной Сибири, самое глубокое озеро на планете.",
        "question": "Где расположено озеро Байкал?",
        "answer": "в южной части Восточной Сибири"
    },
    {
        "context": "Периодическая система химических элементов была впервые опубликована Д. И. Менделеевым в 1869 году.",
        "question": "Кто создал периодическую систему?",
        "answer": "Д. И. Менделеев"
    },
]

SYSTEM_PROMPT_FS = (
    "Ты — система для ответов на вопросы. "
    "Отвечай кратко, извлекая ответ из контекста. "
    "Отвечай только самой сутью ответа, без объяснений."
)

def make_few_shot_prompt(context: str, question: str) -> List[Dict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT_FS}]
    for ex in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": (
            f"Контекст:\n{ex['context']}\n\nВопрос: {ex['question']}\n\nОтвет:"
        )})
        messages.append({"role": "assistant", "content": ex["answer"]})
    messages.append({"role": "user", "content": (
        f"Контекст:\n{context}\n\nВопрос: {question}\n\nОтвет:"
    )})
    return messages


# A/B форматы
def make_prompt_format_a(context: str, question: str) -> List[Dict]:
    """Формат A: XML-теги."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT_ZS},
        {"role": "user", "content": (
            f"<context>{context}</context>\n"
            f"<question>{question}</question>\n"
            f"<answer>"
        )}
    ]

def make_prompt_format_b(context: str, question: str) -> List[Dict]:
    """Формат B: инструкция-стиль."""
    return [
        {"role": "system", "content": (
            "Инструкция: Прочитай текст и найди в нём точный ответ на вопрос. "
            "Выпиши только ответ, без пояснений."
        )},
        {"role": "user", "content": (
            f"Текст: {context}\n\nВопрос: {question}\n\nТвой ответ:"
        )}
    ]

## 6. Движок инференса

In [ ]:
def generate_answer(
    client: OpenAI, model_id: str, messages: List[Dict],
    temperature: float = 0.0, max_tokens: int = 256
) -> Tuple[str, float]:
    """Один запрос к модели. Возвращает (ответ, время в секундах)."""
    start = time.time()
    response = client.chat.completions.create(
        model=model_id, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    elapsed = time.time() - start
    return response.choices[0].message.content.strip(), elapsed


def run_experiment(
    df: pd.DataFrame, model_name: str, prompt_fn,
    temperature: float = 0.0, max_tokens: int = 256,
    experiment_label: str = ""
) -> pd.DataFrame:
    """Прогоняет датасет через модель, считает все метрики. Возвращает DataFrame."""
    config = MODELS[model_name]
    client, model_id = config["client"], config["model_id"]
    
    results = []
    for idx, row in df.iterrows():
        messages = prompt_fn(row["context"], row["question"])
        try:
            raw_answer, gen_time = generate_answer(
                client, model_id, messages, temperature, max_tokens
            )
        except Exception as e:
            print(f"  Ошибка на id={row['id']}: {e}")
            raw_answer, gen_time = "", 0.0
        
        metrics = compute_all_metrics(
            raw_answer, row["reference_answer"], row["all_answers"], gen_time
        )
        metrics["raw_answer"] = raw_answer
        metrics["clean_answer"] = strip_thinking(raw_answer)
        metrics["id"] = row["id"]
        results.append(metrics)
        
        if len(results) % 10 == 0:
            avg_f1 = np.mean([r["f1_score"] for r in results])
            print(f"  [{model_name}] {len(results)}/{len(df)} | avg F1: {avg_f1:.3f}")
    
    result_df = pd.DataFrame(results)
    result_df["model"] = model_name
    result_df["experiment"] = experiment_label
    result_df["temperature"] = temperature
    result_df["prompt_type"] = prompt_fn.__name__
    return result_df

## 7. Эксперименты

### Эксперимент 1: Zero-Shot Baseline

In [14]:
print("=" * 60)
print("Эксперимент 1: Zero-Shot Baseline")
print("=" * 60)

results_zs = {}
for model_name in MODELS:
    print(f"\nЗапускаем {model_name}...")
    results_zs[model_name] = run_experiment(
        df_dataset, model_name, make_zero_shot_prompt,
        temperature=0.0, experiment_label="zero_shot"
    )

# Объединяем результаты и выводим сводку
df_zs = pd.concat(results_zs.values(), ignore_index=True)
print("\n" + "=" * 60)
print("Сводка Zero-Shot:")
print("=" * 60)
print(df_zs.groupby("model")[
    ["exact_match", "f1_score", "bleu", "semantic_similarity", "generation_time"]
].mean().round(4))

Эксперимент 1: Zero-Shot Baseline

Запускаем qwen3.5-9b...


  [qwen3.5-9b] 10/100 | avg F1: 0.719


  [qwen3.5-9b] 20/100 | avg F1: 0.748


  [qwen3.5-9b] 30/100 | avg F1: 0.797


  [qwen3.5-9b] 40/100 | avg F1: 0.814


...

  [gpt-4o] 60/100 | avg F1: 0.813


  [gpt-4o] 70/100 | avg F1: 0.812


  [gpt-4o] 80/100 | avg F1: 0.818


  [gpt-4o] 90/100 | avg F1: 0.807


  [gpt-4o] 100/100 | avg F1: 0.804

Сводка Zero-Shot:
            exact_match  f1_score    bleu  semantic_similarity  \
model                                                            
gpt-4o             0.52    0.8036  0.4128               0.8791   
qwen3.5-9b         0.51    0.8078  0.4276               0.8812   

            generation_time  
model                        
gpt-4o               0.7608  
qwen3.5-9b           0.5032  


### Эксперимент 2: Few-Shot (3 примера)

In [15]:
print("=" * 60)
print("Эксперимент 2: Few-Shot (3 примера)")
print("=" * 60)

results_fs = {}
for model_name in MODELS:
    print(f"\nЗапускаем {model_name}...")
    results_fs[model_name] = run_experiment(
        df_dataset, model_name, make_few_shot_prompt,
        temperature=0.0, experiment_label="few_shot"
    )

df_fs = pd.concat(results_fs.values(), ignore_index=True)
print("\n" + "=" * 60)
print("Сводка Few-Shot:")
print("=" * 60)
print(df_fs.groupby("model")[
    ["exact_match", "f1_score", "bleu", "semantic_similarity", "generation_time"]
].mean().round(4))

# Сравнение zero-shot vs few-shot
print("\n" + "=" * 60)
print("Сравнение: Zero-Shot vs Few-Shot (разница средних F1)")
print("=" * 60)
for model_name in MODELS:
    zs_f1 = results_zs[model_name]["f1_score"].mean()
    fs_f1 = results_fs[model_name]["f1_score"].mean()
    diff = fs_f1 - zs_f1
    print(f"  {model_name}: zero-shot F1={zs_f1:.4f}, few-shot F1={fs_f1:.4f}, "
          f"разница={diff:+.4f} ({'лучше' if diff > 0 else 'хуже'})")

Эксперимент 2: Few-Shot (3 примера)

Запускаем qwen3.5-9b...


  [qwen3.5-9b] 10/100 | avg F1: 0.986


  [qwen3.5-9b] 20/100 | avg F1: 0.940


  [qwen3.5-9b] 30/100 | avg F1: 0.937


  [qwen3.5-9b] 40/100 | avg F1: 0.929


...

  [gpt-4o] 60/100 | avg F1: 0.836


  [gpt-4o] 70/100 | avg F1: 0.833


  [gpt-4o] 80/100 | avg F1: 0.838


  [gpt-4o] 90/100 | avg F1: 0.834


  [gpt-4o] 100/100 | avg F1: 0.834

Сводка Few-Shot:
            exact_match  f1_score    bleu  semantic_similarity  \
model                                                            
gpt-4o             0.58    0.8344  0.6353               0.9095   
qwen3.5-9b         0.65    0.8908  0.7251               0.9376   

            generation_time  
model                        
gpt-4o               0.6947  
qwen3.5-9b           0.5292  

Сравнение: Zero-Shot vs Few-Shot (разница средних F1)
  qwen3.5-9b: zero-shot F1=0.8078, few-shot F1=0.8908, разница=+0.0830 (лучше)
  gpt-4o: zero-shot F1=0.8036, few-shot F1=0.8344, разница=+0.0308 (лучше)


### Эксперимент 3: Влияние длины контекста на качество

Разбиение на 4 квартиля по длине контекста.

In [ ]:
print("=" * 60)
print("Эксперимент 3: Влияние длины контекста на качество")
print("=" * 60)

results_zs_ctx = {}
for model_name, res_df in results_zs.items():
    res_df = res_df.merge(
        df_dataset[["id", "context_length"]], on="id", how="left"
    )
    res_df["context_bin"] = pd.qcut(
        res_df["context_length"], q=4,
        labels=["Короткий", "Средне-короткий", "Средне-длинный", "Длинный"]
    )
    results_zs_ctx[model_name] = res_df

for model_name, res_df in results_zs_ctx.items():
    print(f"\n{model_name}:")
    print(res_df.groupby("context_bin")[
        ["f1_score", "exact_match", "semantic_similarity", "generation_time"]
    ].mean().round(4))

### Эксперимент 4: A/B-тестирование форматов промптов

Три формата: baseline (текст), XML-теги, инструкция-стиль.

In [17]:
print("=" * 60)
print("Эксперимент 4: A/B-тестирование форматов промптов")
print("=" * 60)

prompt_formats = {
    "baseline": make_zero_shot_prompt,
    "format_a_xml": make_prompt_format_a,
    "format_b_instruction": make_prompt_format_b,
}

ab_results = {}
for fmt_name, prompt_fn in prompt_formats.items():
    for model_name in MODELS:
        label = f"{fmt_name}_{model_name}"
        print(f"\nЗапускаем {label}...")
        ab_results[label] = run_experiment(
            df_dataset, model_name, prompt_fn,
            temperature=0.0, experiment_label=fmt_name
        )

df_ab = pd.concat(ab_results.values(), ignore_index=True)
print("\n" + "=" * 60)
print("Сводка A/B теста:")
print("=" * 60)
print(df_ab.groupby(["model", "experiment"])[
    ["f1_score", "exact_match", "bleu", "semantic_similarity"]
].mean().round(4))

Эксперимент 4: A/B-тестирование форматов промптов

Запускаем baseline_qwen3.5-9b...


  [qwen3.5-9b] 10/100 | avg F1: 0.671


  [qwen3.5-9b] 20/100 | avg F1: 0.724


  [qwen3.5-9b] 30/100 | avg F1: 0.780


  [qwen3.5-9b] 40/100 | avg F1: 0.802


...

  [gpt-4o] 60/100 | avg F1: 0.843


  [gpt-4o] 70/100 | avg F1: 0.839


  [gpt-4o] 80/100 | avg F1: 0.855


  [gpt-4o] 90/100 | avg F1: 0.849


  [gpt-4o] 100/100 | avg F1: 0.857

Сводка A/B теста:
                                 f1_score  exact_match    bleu  \
model      experiment                                            
gpt-4o     baseline                0.8129         0.53  0.4204   
           format_a_xml            0.8082         0.52  0.4111   
           format_b_instruction    0.8568         0.56  0.4898   
qwen3.5-9b baseline                0.8138         0.52  0.4285   
           format_a_xml            0.8216         0.53  0.4392   
           format_b_instruction    0.8785         0.60  0.6951   

                                 semantic_similarity  
model      experiment                                 
gpt-4o     baseline                           0.8751  
           format_a_xml                       0.8720  
           format_b_instruction               0.8974  
qwen3.5-9b baseline                           0.8841  
           format_a_xml                       0.8783  
           format_b_instruction 

### Эксперимент 5: Влияние температуры генерации

Значения: 0.0, 0.3, 0.7, 1.0.

In [18]:
print("=" * 60)
print("Эксперимент 5: Влияние температуры генерации")
print("=" * 60)

temperatures = [0.0, 0.3, 0.7, 1.0]
temp_results = {}

for temp in temperatures:
    for model_name in MODELS:
        label = f"temp_{temp}_{model_name}"
        print(f"\nЗапускаем {label}...")
        temp_results[label] = run_experiment(
            df_dataset, model_name, make_zero_shot_prompt,
            temperature=temp, experiment_label=f"temp_{temp}"
        )

df_temp = pd.concat(temp_results.values(), ignore_index=True)
print("\n" + "=" * 60)
print("Сводка по температурам:")
print("=" * 60)
print(df_temp.groupby(["model", "temperature"])[
    ["f1_score", "exact_match", "bleu", "semantic_similarity",
     "response_length_chars"]
].mean().round(4))

Эксперимент 5: Влияние температуры генерации

Запускаем temp_0.0_qwen3.5-9b...


  [qwen3.5-9b] 10/100 | avg F1: 0.719


  [qwen3.5-9b] 20/100 | avg F1: 0.747


  [qwen3.5-9b] 30/100 | avg F1: 0.796


  [qwen3.5-9b] 40/100 | avg F1: 0.814


...

  [gpt-4o] 60/100 | avg F1: 0.768


  [gpt-4o] 70/100 | avg F1: 0.761


  [gpt-4o] 80/100 | avg F1: 0.781


  [gpt-4o] 90/100 | avg F1: 0.773


  [gpt-4o] 100/100 | avg F1: 0.779

Сводка по температурам:
                        f1_score  exact_match    bleu  semantic_similarity  \
model      temperature                                                       
gpt-4o     0.0            0.7886         0.49  0.4049               0.8715   
           0.3            0.8032         0.50  0.4149               0.8711   
           0.7            0.7910         0.49  0.3901               0.8763   
           1.0            0.7794         0.44  0.4076               0.8767   
qwen3.5-9b 0.0            0.8075         0.51  0.4250               0.8811   
           0.3            0.7926         0.49  0.4184               0.8683   
           0.7            0.7826         0.51  0.4036               0.8722   
           1.0            0.7805         0.49  0.4301               0.8732   

                        response_length_chars  
model      temperature                         
gpt-4o     0.0                          31.92  
           0.3 

### Эксперимент 6: Анализ типичных ошибок

Классификация ответов: correct, partial_overlap, verbose, incomplete, completely_wrong.

In [ ]:
def classify_error(prediction: str, reference: str,
                   f1: float, em: float) -> str:
    """Классифицирует тип ошибки модели."""
    pred_norm = normalize_answer_ru(prediction)
    ref_norm = normalize_answer_ru(reference)

    if em == 1.0:
        return "correct"
    if f1 >= 0.8:
        return "partial_overlap"
    if ref_norm in pred_norm:
        return "verbose"
    if pred_norm in ref_norm:
        return "incomplete"
    if f1 > 0:
        return "partial_overlap"
    return "completely_wrong"


for model_name, res_df in results_zs.items():
    merged = res_df.merge(
        df_dataset[["id", "context", "question", "reference_answer"]],
        on="id", how="left"
    )

    print(f"\n{'=' * 60}")
    print(f"ТОП-5 лучших — {model_name}:")
    print(f"{'=' * 60}")
    for _, row in merged.nlargest(5, "f1_score").iterrows():
        print(f"  Q: {row['question']}")
        print(f"  Ref: {row['reference_answer']}")
        print(f"  Pred: {row['clean_answer'][:100]}")
        print(f"  F1={row['f1_score']:.3f}, EM={row['exact_match']:.0f}\n")

    print(f"ХУДШИЕ 5 — {model_name}:")
    print(f"{'=' * 60}")
    for _, row in merged.nsmallest(5, "f1_score").iterrows():
        print(f"  Q: {row['question']}")
        print(f"  Ref: {row['reference_answer']}")
        print(f"  Pred: {row['clean_answer'][:100]}")
        print(f"  F1={row['f1_score']:.3f}, EM={row['exact_match']:.0f}\n")

In [ ]:
print("=" * 60)
print("Распределение типов ошибок:")
print("=" * 60)

error_data = {}
for model_name, res_df in results_zs.items():
    merged = res_df.merge(
        df_dataset[["id", "reference_answer"]], on="id", how="left"
    )
    merged["error_type"] = merged.apply(
        lambda r: classify_error(
            r["clean_answer"], r["reference_answer"],
            r["f1_score"], r["exact_match"]
        ), axis=1
    )
    error_data[model_name] = merged
    print(f"\n{model_name}:")
    print(merged["error_type"].value_counts().to_string())

## 8. Визуализации

In [ ]:
df_all = pd.concat([df_zs, df_fs], ignore_index=True)

summary = df_all.groupby(["model", "experiment"]).agg({
    "exact_match": "mean", "f1_score": "mean", "bleu": "mean",
    "semantic_similarity": "mean", "generation_time": "mean",
    "response_length_chars": "mean", "answer_containment": "mean",
}).round(4)

print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("=" * 80)
display(summary)

### Zero-Shot vs Few-Shot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics_to_plot = [
    ("exact_match", "Exact Match"), ("f1_score", "F1 Score"),
    ("bleu", "BLEU"), ("semantic_similarity", "Semantic Similarity"),
    ("generation_time", "Время генерации (сек)"), ("answer_containment", "Answer Containment"),
]

for ax, (metric, title) in zip(axes.flat, metrics_to_plot):
    pivot = df_all.groupby(["model", "experiment"])[metric].mean().unstack()
    pivot.plot(kind="bar", ax=ax, edgecolor="black")
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(metric)
    ax.legend(title="Промпт")
    ax.tick_params(axis='x', rotation=0)

plt.suptitle("Сравнение моделей: Zero-Shot vs Few-Shot", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("plots/model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

### Влияние температуры

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
temp_metrics = ["f1_score", "exact_match", "response_length_chars"]
temp_titles = ["F1 Score", "Exact Match", "Длина ответа (символы)"]

for ax, metric, title in zip(axes, temp_metrics, temp_titles):
    for model_name in MODELS:
        model_data = df_temp[df_temp["model"] == model_name]
        means = model_data.groupby("temperature")[metric].mean()
        stds = model_data.groupby("temperature")[metric].std()
        ax.errorbar(means.index, means.values, yerr=stds.values,
                    marker='o', label=model_name, capsize=3)
    ax.set_xlabel("Температура")
    ax.set_ylabel(metric)
    ax.set_title(f"Температура vs {title}")
    ax.legend()

plt.tight_layout()
plt.savefig("plots/temperature_impact.png", dpi=150, bbox_inches='tight')
plt.show()

### Длина контекста vs F1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model_name, res_df) in zip(axes, results_zs_ctx.items()):
    ax.scatter(res_df["context_length"], res_df["f1_score"],
               alpha=0.5, edgecolors='black', linewidths=0.5)
    z = np.polyfit(res_df["context_length"], res_df["f1_score"], 1)
    p = np.poly1d(z)
    x_line = np.linspace(res_df["context_length"].min(),
                         res_df["context_length"].max(), 100)
    ax.plot(x_line, p(x_line), "r--", alpha=0.8, label="Тренд")
    ax.set_xlabel("Длина контекста (символы)")
    ax.set_ylabel("F1 Score")
    ax.set_title(f"{model_name}: длина контекста vs F1")
    ax.legend()

plt.tight_layout()
plt.savefig("plots/prompt_length_impact.png", dpi=150, bbox_inches='tight')
plt.show()

### A/B тест форматов

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
pivot_ab = df_ab.groupby(["model", "experiment"])["f1_score"].mean().unstack()
pivot_ab.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("A/B тест: сравнение форматов промптов (F1 Score)")
ax.set_ylabel("Средний F1 Score")
ax.tick_params(axis='x', rotation=0)
ax.legend(title="Формат промпта")
plt.tight_layout()
plt.savefig("plots/ab_test.png", dpi=150, bbox_inches='tight')
plt.show()

### Распределение типов ошибок

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (model_name, merged) in zip(axes, error_data.items()):
    counts = merged["error_type"].value_counts()
    counts.plot(kind="pie", ax=ax, autopct='%1.1f%%', startangle=90)
    ax.set_title(f"{model_name}: типы ошибок")
    ax.set_ylabel("")
plt.tight_layout()
plt.savefig("plots/error_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Статистическая значимость (бонус)

Wilcoxon signed-rank test, paired t-test, bootstrap 95% CI для разницы F1 между моделями.

In [ ]:
print("=" * 60)
print("Статистическая значимость: Qwen 3.5 9B vs GPT-4o")
print("=" * 60)

model_names = list(results_zs.keys())
if len(model_names) >= 2:
    scores_a = results_zs[model_names[0]].sort_values("id")["f1_score"].values
    scores_b = results_zs[model_names[1]].sort_values("id")["f1_score"].values

    print(f"\nСредний F1: {model_names[0]}={scores_a.mean():.4f}, "
          f"{model_names[1]}={scores_b.mean():.4f}")

    # Wilcoxon signed-rank test
    stat, p_value = stats.wilcoxon(scores_a, scores_b)
    print(f"\n--- Wilcoxon signed-rank test ---")
    print(f"  Statistic: {stat:.4f}, p-value: {p_value:.6f}")
    print(f"  Значимо (alpha=0.05): {'Да' if p_value < 0.05 else 'Нет'}")

    # Paired t-test
    t_stat, t_p = stats.ttest_rel(scores_a, scores_b)
    print(f"\n--- Paired t-test ---")
    print(f"  t-statistic: {t_stat:.4f}, p-value: {t_p:.6f}")
    print(f"  Значимо (alpha=0.05): {'Да' if t_p < 0.05 else 'Нет'}")

    # Bootstrap 95% CI
    diffs = scores_a - scores_b
    rng = np.random.RandomState(SEED)
    boot_means = [
        np.mean(rng.choice(diffs, size=len(diffs), replace=True))
        for _ in range(10000)
    ]
    ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])
    print(f"\n--- Bootstrap 95% CI ({model_names[0]} - {model_names[1]}) ---")
    print(f"  [{ci_lower:.4f}, {ci_upper:.4f}]")
    if ci_lower > 0:
        print(f"  Вывод: {model_names[0]} значимо лучше")
    elif ci_upper < 0:
        print(f"  Вывод: {model_names[1]} значимо лучше")
    else:
        print(f"  Вывод: разница не значима (CI содержит 0)")

## 10. Сохранение результатов

In [ ]:
df_final = pd.concat([df_zs, df_fs, df_ab, df_temp], ignore_index=True)
df_final.to_csv("results/all_results.csv", index=False, encoding="utf-8-sig")
print(f"Сохранено {len(df_final)} строк в results/all_results.csv")

summary_report = df_final.groupby(["model", "experiment", "temperature"]).agg({
    "exact_match": ["mean", "std"],
    "f1_score": ["mean", "std"],
    "bleu": ["mean", "std"],
    "semantic_similarity": ["mean", "std"],
    "generation_time": ["mean", "std"],
}).round(4)
summary_report.to_csv("results/summary_report.csv", encoding="utf-8-sig")
print("Сводная таблица: results/summary_report.csv")

display(summary_report)

## 11. Выводы

### Сравнение моделей

Qwen 3.5 9B и GPT-4o показали **эквивалентное качество** на zero-shot QA (F1 0.81 vs 0.80, EM 0.51 vs 0.52). Разница статистически не значима: Wilcoxon p=0.73, paired t-test p=0.87, bootstrap 95% CI содержит 0. При этом Qwen работает быстрее (0.50 vs 0.76 сек) и не требует платного API.

### Zero-shot vs Few-shot

Few-shot даёт прирост обеим моделям, но Qwen выигрывает больше: F1 +0.08 (до 0.89), EM +0.14 (до 0.65). У GPT-4o прирост меньше: F1 +0.03, EM +0.06. Вероятная причина — GPT-4o лучше справляется с форматом ответа без примеров, а Qwen сильнее зависит от демонстрации ожидаемого формата.

### Температура

Обе модели оптимальны при temperature 0.0–0.3. Повышение до 1.0 снижает F1 на ~0.03. Эффект слабый, но монотонный — для extractive QA стохастичность генерации вредна.

### A/B тест форматов

Format B (инструкция-стиль) — лучший: Qwen F1=0.88 (+0.07 vs baseline), GPT-4o F1=0.86 (+0.05). XML-теги (Format A) не дают значимого улучшения. Вывод: явная формулировка задачи («Прочитай текст и найди ответ») эффективнее формальной структуры.

### Анализ ошибок

Распределение типов ошибок схоже: correct ~51-52%, partial_overlap ~29-33%, completely_wrong ~5%. Qwen чаще verbose (11% vs 8%), GPT-4o чаще partial_overlap (33% vs 29%). Обе модели редко дают полностью неверные ответы — основная проблема в формулировке, а не в понимании.

### Информативность метрик

- **F1 Score** — наиболее информативная метрика, хорошо дифференцирует качество ответов.
- **Semantic Similarity** — полезна как дополнение, но слабо дифференцирует (разброс 0.76–0.94 у обеих моделей). Может завышать оценку тематически близких, но фактически неверных ответов.
- **BLEU** — наименее информативна для коротких extractive ответов (создана для оценки перевода).
- **Exact Match** — слишком бинарная, не учитывает частично правильные ответы.
- **Answer Containment** (кастомная) — полезна для выявления verbose-ответов, компенсирует слабость EM и F1 на многословных ответах.

### Рекомендации

- Для русскоязычного extractive QA модели класса 9B (Qwen 3.5) достаточно — платный API не даёт преимуществ.
- Оптимальная конфигурация: few-shot + формат-инструкция + temperature=0.
- При выборе бенчмарков для QA следует опираться на F1 Score как основную метрику, дополняя Semantic Similarity и Answer Containment.

### Ограничения

- Выборка 100 примеров достаточна для грубого сравнения, но не для тонких различий (подтверждено: разница моделей не значима).
- Время генерации включает сетевую задержку (GPT-4o) и не является чистым показателем скорости инференса.
- Qwen в квантизации Q4_K_M — полная модель может показать результаты выше.